# Pre-season points model

Predict a player's early-season average fixture points from aggregates
for the preceding season.

## Setup

In [ ]:
import numpy as np
import pandas as pd
from catboost import CatBoostRegressor
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import train_test_split

from prediction.artifacts.io import PRE_SEASON_ARTIFACT_PATH, save_trained_catboost_model
from training.config import EARLY_GAMEWEEKS, NUMERIC_FEATURES, RANDOM_STATE
from training.load_training_data import load_historic_player_fixture_data

TRAINING_SEASONS = ["2021-22", "2022-23", "2023-24"]
TARGET_COLUMN = "target_avg_fixture_points_first_gws"
VALIDATION_FRACTION = 0.20
CATEGORICAL_COLUMNS = ["position"]
CATBOOST_PARAMS = {
    "iterations": 163,
    "learning_rate": 0.03,
    "depth": 6,
    "loss_function": "RMSE",
    "random_seed": RANDOM_STATE,
    "verbose": False,
    "allow_writing_files": False,
}

## Load data

In [ ]:
fixture_history_df = pd.concat(
    [
        load_historic_player_fixture_data(season).assign(season=season)
        for season in TRAINING_SEASONS
    ],
    ignore_index=True,
    sort=False,
).sort_values(["season", "name", "GW", "fixture_id"])
fixture_history_df[NUMERIC_FEATURES] = fixture_history_df[NUMERIC_FEATURES].apply(
    pd.to_numeric, errors="coerce"
)
print(fixture_history_df.groupby("season").size())

## Engineer features and target

In [ ]:
player_season_categories = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[
        CATEGORICAL_COLUMNS
    ].last()
)
player_season_totals = (
    fixture_history_df.groupby(["season", "name"], as_index=False)[NUMERIC_FEATURES]
    .sum(min_count=1)
    .rename(columns={feature: f"season_sum_{feature}" for feature in NUMERIC_FEATURES})
)
player_season_features = player_season_categories.merge(
    player_season_totals,
    on=["season", "name"],
    validate="one_to_one",
)

early_fixture_targets = (
    fixture_history_df.loc[fixture_history_df["GW"].between(1, EARLY_GAMEWEEKS)]
    .groupby(["season", "name"], as_index=False)
    .agg(**{TARGET_COLUMN: ("total_points", "mean")})
    .rename(columns={"season": "target_season"})
)
subsequent_season = dict(zip(TRAINING_SEASONS, TRAINING_SEASONS[1:]))
player_season_features["target_season"] = player_season_features["season"].map(
    subsequent_season
)
model_data = (
    player_season_features.dropna(subset=["target_season"])
    .merge(
        early_fixture_targets,
        on=["target_season", "name"],
        validate="one_to_one",
    )
    .sort_values(["season", "name"])
    .reset_index(drop=True)
)
model_features = CATEGORICAL_COLUMNS + [
    f"season_sum_{feature}" for feature in NUMERIC_FEATURES
]

assert not model_data.duplicated(["season", "name"]).any()
print(model_data.groupby(["season", "target_season"]).size())

## Split training and validation data

In [ ]:
train_index, valid_index = train_test_split(
    model_data.index,
    test_size=VALIDATION_FRACTION,
    random_state=RANDOM_STATE,
)
X_train = model_data.loc[train_index, model_features].copy()
X_valid = model_data.loc[valid_index, model_features].copy()
y_train = model_data.loc[train_index, TARGET_COLUMN].copy()
y_valid = model_data.loc[valid_index, TARGET_COLUMN].copy()

for frame in (X_train, X_valid):
    frame[CATEGORICAL_COLUMNS] = (
        frame[CATEGORICAL_COLUMNS].fillna("__MISSING__").astype(str)
    )

print(f"Training rows: {len(X_train):,}; validation rows: {len(X_valid):,}")

## Train candidate models

In [ ]:
validation_predictions = {}

dummy_model = DummyRegressor(strategy="mean").fit(X_train, y_train)
validation_predictions["Dummy"] = dummy_model.predict(X_valid)

pre_season_model = CatBoostRegressor(**CATBOOST_PARAMS).fit(
    X_train, y_train, cat_features=CATEGORICAL_COLUMNS
)
validation_predictions["CatBoost"] = pre_season_model.predict(X_valid)

## Evaluate models

In [ ]:
def evaluate_predictions(predictions, top_fraction=0.10):
    evaluation = pd.DataFrame({"actual": y_valid, "predicted": predictions})
    count = max(1, int(np.ceil(len(evaluation) * top_fraction)))
    predicted_top = evaluation.nlargest(count, "predicted")
    actual_top_indices = set(evaluation.nlargest(count, "actual").index)
    return {
        "MAE": mean_absolute_error(evaluation["actual"], evaluation["predicted"]),
        "RMSE": root_mean_squared_error(evaluation["actual"], evaluation["predicted"]),
        "R2": r2_score(evaluation["actual"], evaluation["predicted"]),
        "top_decile_avg_actual_points": predicted_top["actual"].mean(),
        "top_decile_hit_rate": predicted_top.index.isin(actual_top_indices).mean(),
        "top_decile_oracle_regret": (
            evaluation.nlargest(count, "actual")["actual"].mean()
            - predicted_top["actual"].mean()
        ),
    }

comparison_table = (
    pd.DataFrame.from_dict(
        {
            name: evaluate_predictions(predictions)
            for name, predictions in validation_predictions.items()
        },
        orient="index",
    )
    .rename_axis("model")
    .reset_index()
    .sort_values(["top_decile_avg_actual_points", "MAE"], ascending=[False, True])
    .reset_index(drop=True)
)
comparison_table.round(3)

## Export selected model

In [ ]:
save_trained_catboost_model(
    model=pre_season_model,
    feature_columns=model_features,
    categorical_columns=CATEGORICAL_COLUMNS,
    model_name="Unweighted CatBoost pre-season model",
    model_version="1.0",
    save_path=PRE_SEASON_ARTIFACT_PATH,
)
print(f"Saved pre-season model to {PRE_SEASON_ARTIFACT_PATH}")